In [1]:
# Import required Python package
!pip install pandas

# Install Node.js (because tweet-harvest built using Node.js)
!sudo apt-get update
!sudo apt-get install -y ca-certificates curl gnupg
!sudo mkdir -p /etc/apt/keyrings
!curl -fsSL https://deb.nodesource.com/gpgkey/nodesource-repo.gpg.key | sudo gpg --dearmor -o /etc/apt/keyrings/nodesource.gpg

!NODE_MAJOR=20 && echo "deb [signed-by=/etc/apt/keyrings/nodesource.gpg] https://deb.nodesource.com/node_$NODE_MAJOR.x nodistro main" | sudo tee /etc/apt/sources.list.d/nodesource.list

!sudo apt-get update
!sudo apt-get install nodejs -y

!node -v


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
curl: Failed writing body
'NODE_MAJOR' is not recognized as an internal or external command,
operable program or batch file.
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app


v24.12.0


In [ ]:
import pandas as pd
import time
import os

# 1. MASUKKAN AUTH TOKEN TUMBAL LO DI SINI
twitter_auth_token = '8bfff0f8a261f5020bd9b3017cedeb3dacac9ee3' 

# 2. Batas per emosi agar seimbang
limit = 1000

# 3. Kueri spesifik seperti yang kita bahas sebelumnya
queries = {
    "anger": '"gue marah banget" OR "emosi banget" OR "bikin emosi" lang:id',
    "joy": '"seneng banget" OR "akhirnya kesampaian" OR "alhamdulillah" lang:id',
    "sadness": '"nangis banget" OR "nyesek" OR "sedih banget" lang:id',
    "fear": '"takut banget" OR "merinding" OR "parno" lang:id',
    "love": '"sayang banget" OR "cinta banget" OR "gemes" lang:id'
}

print("Memulai proses scraping berantai...")

for emotion, keyword in queries.items():
    print(f"\n=======================================================")
    print(f"MENARIK DATA EMOSI: {emotion.upper()}")
    print(f"=======================================================")
    
    filename = f"emosi_{emotion}.csv"
    
    # Menjalankan perintah tweet-harvest lewat command line Colab
    command = f'npx -y tweet-harvest@2.6.1 -o "{filename}" -s "{keyword}" --tab "LATEST" -l {limit} --token {twitter_auth_token}'
    os.system(command)
    
    print(f"\n[INFO] {emotion} selesai. Jeda 30 detik untuk menghindari limit API...")
    time.sleep(30) # JANGAN DIHAPUS. Ini agar akun tumbal lo nggak langsung diblokir.

print("\nSELURUH PROSES SCRAPING SELESAI!")

Memulai proses scraping berantai...

MENARIK DATA EMOSI: ANGER

[INFO] anger selesai. Jeda 30 detik untuk menghindari limit API...

MENARIK DATA EMOSI: JOY

[INFO] joy selesai. Jeda 30 detik untuk menghindari limit API...

MENARIK DATA EMOSI: SADNESS

[INFO] sadness selesai. Jeda 30 detik untuk menghindari limit API...

MENARIK DATA EMOSI: FEAR

[INFO] fear selesai. Jeda 30 detik untuk menghindari limit API...

MENARIK DATA EMOSI: LOVE

[INFO] love selesai. Jeda 30 detik untuk menghindari limit API...

SELURUH PROSES SCRAPING SELESAI!


: 

In [ ]:
import pandas as pd
import glob

print("Menggabungkan file CSV...")

# Mencari semua file yang berawalan 'emosi_' di dalam folder tweets-data/
file_paths = glob.glob("tweets-data/emosi_*.csv")
all_dfs = []

for path in file_paths:
    # Nama file adalah format: tweets-data/emosi_anger.csv
    # Kita ekstrak nama emosinya dari nama file
    emotion_label = path.split('_')[1].split('.')[0] 
    
    df = pd.read_csv(path)
    # Tambahkan kolom label yang sangat penting untuk Deep Learning lo
    df['emotion_class'] = emotion_label 
    
    # Kita hanya ambil kolom yang relevan: teks dan labelnya
    # (tweet-harvest menyimpan teks di kolom 'full_text')
    df_clean = df[['full_text', 'emotion_class']].rename(columns={'full_text': 'text', 'emotion_class': 'label'})
    
    all_dfs.append(df_clean)

# Gabungkan semuanya
final_df = pd.concat(all_dfs, ignore_index=True)

# Acak urutan barisnya agar model tidak bias saat dilatih
final_df = final_df.sample(frac=1).reset_index(drop=True)

# Simpan hasil akhir
output_file = "dataset_5_emosi_twitter.csv"
final_df.to_csv(output_file, index=False)

print(f"\nPenggabungan selesai! Dataset lo tersimpan di '{output_file}'.")
print("Distribusi kelas yang berhasil didapat:")
print(final_df['label'].value_counts())